# Báo cáo chất lượng dữ liệu tự động cho Olist

Bạn sắp giao bộ Olist cho cả team dùng (EDA, model, dashboard). Trước đó, viết một báo cáo chất
lượng chạy lại được mỗi lần dữ liệu về: 15 kiểm tra có tên, có mức độ, có mẫu số rõ ràng.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm và hằng.
- Cell gắn tag `explore` (dòng đầu `# explore`) đọc dữ liệu thật và gọi hàm để xem. Test bỏ qua chúng.
- Chấm: `uv run pytest tracks/data-science/modules/09-eda-storytelling/exercises/intermediate/01-olist-quality-report -q`

In [ ]:
import numpy as np
import pandas as pd

# (tên kiểm tra, bảng, severity) — đúng thứ tự này trong báo cáo
CHECKS = [
    ("orders_pk_duplicated", "orders", "error"),
    ("delivered_missing_delivery_date", "orders", "warning"),
    ("delivered_missing_approved_at", "orders", "warning"),
    ("carrier_after_customer_delivery", "orders", "error"),
    ("approved_before_purchase", "orders", "error"),
    ("orders_without_items", "orders", "warning"),
    ("items_orphan_order", "items", "error"),
    ("items_orphan_product", "items", "error"),
    ("orders_without_review", "orders", "warning"),
    ("review_id_duplicated", "reviews", "warning"),
    ("products_missing_category", "products", "warning"),
    ("category_not_translated", "products", "warning"),
    ("customer_zip_not_in_geolocation", "customers", "warning"),
    ("payment_items_mismatch", "orders", "warning"),
    ("freight_zero", "items", "info"),
]
REPORT_COLUMNS = ["check", "table", "severity", "n_failed", "n_total", "fail_rate"]

In [ ]:
# explore: đọc 8 bảng Olist thật (tải bằng: uv run python data/download_olist.py)
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data" / "raw"
order_dates = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
               "order_delivered_customer_date", "order_estimated_delivery_date"]
tables = {
    "orders": pd.read_csv(RAW / "olist_orders_dataset.csv", parse_dates=order_dates),
    "items": pd.read_csv(RAW / "olist_order_items_dataset.csv"),
    "payments": pd.read_csv(RAW / "olist_order_payments_dataset.csv"),
    "reviews": pd.read_csv(RAW / "olist_order_reviews_dataset.csv"),
    "products": pd.read_csv(RAW / "olist_products_dataset.csv"),
    "customers": pd.read_csv(RAW / "olist_customers_dataset.csv", dtype={"customer_zip_code_prefix": str}),
    "geolocation": pd.read_csv(RAW / "olist_geolocation_dataset.csv", dtype={"geolocation_zip_code_prefix": str}),
    "translation": pd.read_csv(RAW / "product_category_name_translation.csv"),
}
{name: len(df) for name, df in tables.items()}

## 1. Ba khối xây dựng

Mọi kiểm tra trong báo cáo đều là một trong ba dạng: khoá lặp, khoá ngoại mồ côi, thứ tự thời gian sai.

In [ ]:
def count_duplicated(df: pd.DataFrame, key: list[str]) -> int:
    """Số dòng có key xuất hiện hơn một lần (đếm mọi bản, không chỉ bản thừa)."""
    return int(df.duplicated(subset=key, keep=False).sum())


def count_orphans(child: pd.DataFrame, child_key: str, parent: pd.DataFrame, parent_key: str) -> int:
    """Số dòng của child có child_key khác null nhưng không có trong parent[parent_key]."""
    keys = child[child_key]
    return int((keys.notna() & ~keys.isin(parent[parent_key])).sum())


def count_order_violations(df: pd.DataFrame, earlier: str, later: str) -> tuple[int, int]:
    """(số dòng vi phạm, số dòng được kiểm). Chỉ kiểm dòng có cả hai cột khác null;
    vi phạm khi df[earlier] > df[later]."""
    both = df[earlier].notna() & df[later].notna()
    bad = both & (df[earlier] > df[later])
    return int(bad.sum()), int(both.sum())

In [ ]:
# explore: thử từng hàm trên dữ liệu thật
print(count_duplicated(tables["reviews"], ["review_id"]))
print(count_orphans(tables["orders"], "order_id", tables["items"], "order_id"))
print(count_order_violations(tables["orders"], "order_delivered_carrier_date", "order_delivered_customer_date"))

## 2. Báo cáo đủ 15 kiểm tra

Bảng định nghĩa `n_failed` / `n_total` từng kiểm tra nằm trong README của bài.

In [ ]:
def quality_report(t: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Chạy đủ 15 kiểm tra trong CHECKS theo đúng thứ tự; trả DataFrame cột REPORT_COLUMNS.

    t có các khoá: orders, items, payments, reviews, products, customers, geolocation, translation.
    Cột ngày của orders đã là datetime. n_failed / n_total của từng kiểm tra: bảng trong README.
    fail_rate = n_failed / n_total (0.0 nếu n_total = 0).
    """
    o, it, pay, rv = t["orders"], t["items"], t["payments"], t["reviews"]
    pr, cu, geo, tr = t["products"], t["customers"], t["geolocation"], t["translation"]
    delivered = o[o.order_status == "delivered"]
    with_category = pr[pr.product_category_name.notna()]

    # tổng theo đơn ở từng bảng trước, rồi mới ghép: tránh nhân dòng item × payment
    pay_sum = pay.groupby("order_id").payment_value.sum()
    item_sum = (it.price + it.freight_value).groupby(it.order_id).sum()
    both = pd.concat([pay_sum, item_sum], axis=1, keys=["pay", "items"], join="inner")

    results = {
        "orders_pk_duplicated": (count_duplicated(o, ["order_id"]), len(o)),
        "delivered_missing_delivery_date": (int(delivered.order_delivered_customer_date.isna().sum()), len(delivered)),
        "delivered_missing_approved_at": (int(delivered.order_approved_at.isna().sum()), len(delivered)),
        "carrier_after_customer_delivery": count_order_violations(
            o, "order_delivered_carrier_date", "order_delivered_customer_date"),
        # duyệt trước lúc đặt = purchase_timestamp > approved_at
        "approved_before_purchase": count_order_violations(o, "order_purchase_timestamp", "order_approved_at"),
        "orders_without_items": (count_orphans(o, "order_id", it, "order_id"), len(o)),
        "items_orphan_order": (count_orphans(it, "order_id", o, "order_id"), len(it)),
        "items_orphan_product": (count_orphans(it, "product_id", pr, "product_id"), len(it)),
        "orders_without_review": (count_orphans(o, "order_id", rv, "order_id"), len(o)),
        "review_id_duplicated": (count_duplicated(rv, ["review_id"]), len(rv)),
        "products_missing_category": (int(pr.product_category_name.isna().sum()), len(pr)),
        "category_not_translated": (
            count_orphans(with_category, "product_category_name", tr, "product_category_name"), len(with_category)),
        "customer_zip_not_in_geolocation": (
            count_orphans(cu, "customer_zip_code_prefix", geo, "geolocation_zip_code_prefix"), len(cu)),
        "payment_items_mismatch": (int(((both.pay - both["items"]).abs() > 1.0).sum()), len(both)),
        "freight_zero": (int((it.freight_value == 0).sum()), len(it)),
    }
    rows = []
    for name, table, severity in CHECKS:
        n_failed, n_total = results[name]
        rows.append({"check": name, "table": table, "severity": severity, "n_failed": n_failed,
                     "n_total": n_total, "fail_rate": n_failed / n_total if n_total else 0.0})
    return pd.DataFrame(rows, columns=REPORT_COLUMNS)


def blocking_checks(report: pd.DataFrame) -> list[str]:
    """Tên các kiểm tra severity "error" có n_failed > 0, giữ thứ tự trong report."""
    return report.loc[(report.severity == "error") & (report.n_failed > 0), "check"].tolist()

In [ ]:
# explore: báo cáo trên Olist thật
report = quality_report(tables)
report.style.format({"fail_rate": "{:.4%}"})

## 3. Kiểm tra nào chặn pipeline?

In [ ]:
# explore
blocking_checks(report)

In [ ]:
# explore: 23 đơn có timeline không thể xảy ra trông thế nào?
o = tables["orders"]
o[o.order_delivered_carrier_date > o.order_delivered_customer_date][
    ["order_id", "order_status", "order_purchase_timestamp", "order_delivered_carrier_date",
     "order_delivered_customer_date"]].head()

## 4. Đọc báo cáo (bài mẫu)

- **Chặn**: `carrier_after_customer_delivery` có 23 đơn hãng vận chuyển nhận hàng **sau** khi khách
  đã nhận. Không thể xảy ra, nên ít nhất một trong hai timestamp sai. Mọi phân tích dùng "thời gian
  từ seller tới hãng vận chuyển" phải loại 23 đơn này (và ghi lại), phân tích chỉ dùng ngày khách
  nhận thì vẫn dùng được. Báo cho đội dữ liệu nguồn.
- **Không lỗi**: khoá chính `orders`, item mồ côi, duyệt trước lúc đặt đều bằng 0. Đây cũng là thông
  tin: join `items → orders` và `items → products` an toàn.
- **Cảnh báo có hệ quả**: 775 đơn không có item (phần lớn `unavailable`/`canceled`) → tính doanh thu
  từ `items` sẽ tự loại chúng, tính số đơn từ `orders` thì không: hai con số "số đơn" sẽ lệch nhau.
  1 603 dòng review có `review_id` lặp → dedupe theo `order_id`, không theo `review_id`.
  2 category chưa dịch → join sang tiếng Anh ra NaN cho các sản phẩm đó.
- **Thông tin**: 383 item phí vận chuyển 0 (khuyến mãi miễn phí vận chuyển), không phải lỗi.

Báo cáo này nên chạy tự động mỗi lần nạp dữ liệu, và `blocking_checks` khác rỗng thì dừng pipeline
hoặc ít nhất gắn cờ trên dashboard.